In [ ]:
import tempfile
import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()

import tempfile

def load_pdf_into_vectorstore(file: tempfile) -> str:
    file_path = file.name
    loader = PyPDFLoader(file_path=file_path)
    documents = loader.load()
    text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50, separator="\n")
    docs = text_splitter.split_documents(documents=documents)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(
        docs, embedding=embeddings, persist_directory="chromadb11"
    )

    return 'Document uploaded and index created successfully. You can chat now.'


In [ ]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash"
)

def getresponse(query, history:list) -> tuple:
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = Chroma(
        persist_directory="chromadb11", embedding_function=embeddings
    )

    message = """
                Answer this question using the provided context . If information is not available in the context,
                Just respond saying "I don't know"

                Input: {input}
                Context: {context}
            """

    prompt = ChatPromptTemplate.from_messages([("human", message)])
    llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(vectorstore.as_retriever(), question_answer_chain)
    response = rag_chain.invoke({"input": query})
    history.append((query, response['answer']))

    return "", history

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    with gr.Row():
        with gr.Column():
            file = gr.components.File(  
                label='Upload your pdf file',
                file_count='single',
                file_types=['.pdf'])
            
            upload = gr.components.Button(
                value='Upload', variant='primary')
            
            label = gr.components.Textbox()
            chatbot = gr.Chatbot(label='Talk to the Document')
            msg = gr.Textbox()
            
    upload.click(load_pdf_into_vectorstore, [file], [label])
    msg.submit(getresponse,[msg,chatbot],[msg,chatbot])
    
demo.launch(debug=True)